# Demostración sintética del núcleo hidráulico

Este notebook ejecuta los escenarios 11.1 y 11.2 del diseño v0.2 con el
paquete `modelo_fisico` y representa sus resultados.

**Aviso:** todos los parámetros son sintéticos. El ejemplo no representa
una parcela calibrada ni constituye una recomendación de riego. No usa
datos privados, Drive ni claves de API.

In [ ]:
import subprocess
import sys
from pathlib import Path

URL = "https://github.com/irr1008/TFG_PATATAS.git"
REPO = Path("/content/TFG_PATATAS")

if (REPO / ".git").is_dir():
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
else:
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)

sys.path.insert(0, str(REPO / "src"))

import modelo_fisico

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO, capture_output=True, text=True, check=True,
).stdout.strip()

print("Paquete cargado desde:", modelo_fisico.__file__)
print("Commit:", commit)

In [ ]:
import matplotlib.pyplot as plt
from modelo_fisico import verificacion_temporal as vt

CARPETA_FIGURAS = Path("/content/figuras_demostracion")
CARPETA_FIGURAS.mkdir(parents=True, exist_ok=True)

perfil = vt.crear_perfil()
ejecuciones = vt.ejecutar_escenarios(perfil)
series = {
    clave: vt.extraer_serie(resultado, perfil)
    for clave, resultado in ejecuciones.items()
}

print("Perfil sintético:")
for capa in perfil.capas:
    print(" ", capa)
print("Ejecuciones:", len(ejecuciones))

## Escenario 11.1: aportes y extracción

Riego prescrito durante las dos primeras horas y extracción solicitada
en las dos capas durante el resto de las 24 h. El escenario comprueba la
integración de entradas por intervalos y el cierre del balance.
No hay salida superficial ni drenaje inferior en este caso.

In [ ]:
eventos = vt.crear_eventos_11_1()
x_h = [e.inicio_s / 3600 for e in eventos] + [eventos[-1].fin_s / 3600]

riego = [e.riego_mm_h for e in eventos] + [eventos[-1].riego_mm_h]
ext1 = [e.extracciones_mm_h[0] for e in eventos] + [eventos[-1].extracciones_mm_h[0]]
ext2 = [e.extracciones_mm_h[1] for e in eventos] + [eventos[-1].extracciones_mm_h[1]]

fig, ejes = plt.subplots(3, 1, figsize=(9, 11), sharex=True)

linea_riego, = ejes[0].step(x_h, riego, where="post", color="tab:blue",
                            label="Riego (mm/h), eje izquierdo")
ejes[0].set_ylabel("Riego (mm/h)")
eje_ext = ejes[0].twinx()
linea_e1, = eje_ext.step(x_h, ext1, where="post", color="tab:green",
                         label="Extracción solicitada, capa 1 (mm/h), eje derecho")
linea_e2, = eje_ext.step(x_h, ext2, where="post", color="tab:red",
                         label="Extracción solicitada, capa 2 (mm/h), eje derecho")
eje_ext.set_ylabel("Extracción (mm/h)")
ejes[0].set_title("Escenario 11.1 (sintético): entradas prescritas")
fig.legend(handles=[linea_riego, linea_e1, linea_e2],
           loc="upper center", bbox_to_anchor=(0.5, 0.995),
           ncol=1, fontsize=9)

for paso, estilo in ((300.0, "--"), (37.5, "-")):
    serie = series[("11.1", paso)]
    t_h = [f["tiempo_s"] / 3600 for f in serie]
    ejes[1].plot(t_h, [f["theta1"] for f in serie], estilo,
                 color="tab:blue", label=f"theta1, paso {paso:g} s")
    ejes[1].plot(t_h, [f["theta2"] for f in serie], estilo,
                 color="tab:orange", label=f"theta2, paso {paso:g} s")
ejes[1].set_ylabel("Humedad volumétrica (m³/m³)")
ejes[1].legend()

serie = series[("11.1", 37.5)]
t_h = [f["tiempo_s"] / 3600 for f in serie]
ejes[2].plot(t_h, [f["RO_acumulada_mm"] for f in serie],
             label="Salida superficial acumulada, RO (mm)")
ejes[2].plot(t_h, [f["DP_acumulado_mm"] for f in serie], "--",
             label="Drenaje inferior acumulado, DP (mm)")
ejes[2].set_ylim(-0.05, 0.05)
ejes[2].text(0.5, 0.75, "RO y DP son cero en este escenario",
             transform=ejes[2].transAxes, ha="center")
ejes[2].set_ylabel("Salidas acumuladas (mm)")
ejes[2].set_xlabel("Tiempo (h)")
ejes[2].legend(loc="lower right")

fig.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig(CARPETA_FIGURAS / "escenario_11_1.png", dpi=150)
plt.show()

Las curvas de los distintos pasos casi coinciden: entre 75 y 37,5 s la
diferencia máxima de humedad es de unos 8,5e-6 m³/m³, muy por debajo del
umbral de 1e-4 m³/m³ fijado en el diseño.

## Balance global y refinamiento temporal

El residual global se calcula con los estados extremos y los flujos
acumulados, independientemente de los residuales por paso.
El cierre de masa no basta para aceptar un paso temporal: por eso se
compara también la dependencia del resultado respecto al paso.

In [ ]:
resumen = vt.tabla_resumen(ejecuciones)

print(f"{'Esc.':<6}{'Paso (s)':>9}{'Pasos':>7}"
      f"{'|Residual| (mm)':>17}{'Tolerancia (mm)':>17}  Estado")
for f in resumen:
    residual = abs(f["residual_global_mm"])
    estado = "OK" if residual <= f["tolerancia_global_mm"] else "FALLA"
    print(f"{f['escenario']:<6}{f['paso_s']:>9g}{f['numero_pasos']:>7}"
          f"{residual:>17.2e}{f['tolerancia_global_mm']:>17.2e}  {estado}")
    assert estado == "OK"

refinamiento = vt.tabla_refinamiento_11_1(series)
criterios = vt.tabla_criterios_11_1(refinamiento)

print("\nCriterios del escenario 11.1 (75 s frente a 37,5 s):")
for c in criterios:
    print(f"  {c['criterio']}: {c['valor']:.3e} <= {c['umbral']:g}"
          f" -> {c['cumple']}")
assert all(c["cumple"] for c in criterios)

## Escenario 11.2: drenaje acoplado frente a la solución analítica

Sin aportes ni extracciones. La referencia analítica representa el sistema
continuo de drenaje acoplado; no reproduce el algoritmo secuencial.
Se espera que el error disminuya al reducir el paso, sin exigir
coincidencia exacta para un paso finito.

In [ ]:
esp2 = 1000.0 * perfil.capas[1].espesor_m
t_ref = [k * vt.DURACION_S / 240 for k in range(241)]
errores = vt.tabla_errores_11_2(series, perfil)

fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5))

ejes[0].plot([t / 3600 for t in t_ref],
             [vt.referencia_11_2(t)[1] for t in t_ref],
             "k-", label="Referencia analítica")
for paso in (300.0, 37.5):
    serie = series[("11.2", paso)]
    ejes[0].plot([f["tiempo_s"] / 3600 for f in serie],
                 [f["S2_mm"] for f in serie], "--",
                 label=f"Numérico, paso {paso:g} s")
ejes[0].set_xlabel("Tiempo (h)")
ejes[0].set_ylabel("S2, almacenamiento de la capa 2 (mm)")
ejes[0].set_title("Capa 2")
ejes[0].legend()

for paso in vt.PASOS_S:
    serie = series[("11.2", paso)]
    ejes[1].plot(
        [f["tiempo_s"] / 3600 for f in serie],
        [(f["S2_mm"] - vt.referencia_11_2(f["tiempo_s"])[1]) / esp2
         for f in serie],
        label=f"paso {paso:g} s",
    )
ejes[1].set_xlabel("Tiempo (h)")
ejes[1].set_ylabel("Error de theta2 (m³/m³)")
ejes[1].set_title("Error numérico menos analítico")
ejes[1].legend()

ejes[2].loglog([e["paso_s"] for e in errores],
               [e["max_error_theta2"] for e in errores], "o-")
ejes[2].set_xlabel("Paso temporal (s)")
ejes[2].set_ylabel("Máximo error de theta2 (m³/m³)")
ejes[2].set_title("Dependencia del paso")

fig.tight_layout()
fig.savefig(CARPETA_FIGURAS / "escenario_11_2.png", dpi=150)
plt.show()

print("Paso (s)   Máx. error theta2   Máx. error DP (mm)   Cociente")
for e in errores:
    cociente = e["cociente_error_theta2"]
    cociente = "-" if cociente == "" else f"{cociente:.4f}"
    print(f"{e['paso_s']:>8g}   {e['max_error_theta2']:>16.4e}   "
          f"{e['max_error_DP_mm']:>17.6f}   {cociente}")

## Coherencia con el informe y regeneración de tablas

Las comprobaciones siguientes fallan si los resultados dejan de coincidir
con `docs/verificacion-nucleo-hidraulico.md`. La última celda escribe las
tablas CSV en `/content/resultados_demostracion`.

In [ ]:
assert round(errores[0]["max_error_theta2"], 6) == 0.000105
assert round(errores[3]["max_error_theta2"], 6) == 0.000013
assert round(errores[0]["max_error_DP_mm"], 6) == 0.041812
assert all(
    fino["max_error_theta2"] < grueso["max_error_theta2"]
    for grueso, fino in zip(errores, errores[1:])
)
print("Los resultados coinciden con docs/verificacion-nucleo-hidraulico.md.")

rutas = vt.generar_tablas("/content/resultados_demostracion", commit=commit)
print(len(rutas), "archivos en /content/resultados_demostracion")
for ruta in rutas:
    print(" -", ruta.name)

## Qué comprueba y qué no demuestra

**Comprueba:**
- Que el balance global cierra en las ocho ejecuciones.
- Que las entradas prescritas se integran correctamente por intervalos.
- Que, en el escenario 11.1, la diferencia entre 75 y 37,5 s cumple
  los umbrales del diseño.
- Que, en el escenario 11.2, el error frente a la solución analítica
  disminuye aproximadamente a la mitad al dividir el paso por dos.

**No demuestra:**
- Calibración ni validación con parcelas reales.
- Una precisión válida para cualquier configuración o evento.
- Que el paso de 300 s sea adecuado fuera de estos escenarios.
- Ninguna recomendación de riego.

Los parámetros, estados y entradas son sintéticos. Las limitaciones
completas están en `docs/verificacion-nucleo-hidraulico.md`.